In [1]:
# 04_perturbation.ipynb -- build P1-P5 perturbed variants (incl. P3 factual error and P4 direction flip)

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 04: construct controlled perturbations; carry fin_profile/context for grounding in 05 ---
# P1 rating-rationale inconsistency | P2 remove qualitative support | P3 factual (quantitative) error
# | P4 direction flip (reverse support polarity, grade unchanged) | P5 verbosity control.
src = PROC / "generations"
if not df_exists(src):
    print("[skip] run 03_generation first (no generations).")
else:
    import re
    gen = read_df(src)

    def shift_rating(r, notches):
        k = to_rank(r)
        if np.isnan(k):
            return r
        return RATING_SCALE[int(np.clip(k + notches, 0, len(RATING_SCALE) - 1))]

    SUPPORT_KW = ("support", "parent", "group", "government", "guarant",
                  "owner", "subsidiar", "affiliat", "sponsor")
    # Antonym pairs for the direction flip; applied as whole-word swaps (case-insensitive).
    POLARITY = [("strong", "weak"), ("strength", "weakness"), ("upgrade", "downgrade"),
                ("upward", "downward"), ("above", "below"), ("benefit", "burden"),
                ("favorable", "unfavorable"), ("improving", "deteriorating"),
                ("positive", "negative"), ("uplift", "drag"), ("higher", "lower"),
                ("supports", "undermines"), ("mitigates", "aggravates")]
    _flip = {}
    for a, b in POLARITY:
        _flip[a] = b; _flip[b] = a

    def flip_direction(txt):
        # Reverse the support logic by swapping polarity words; the grade is left unchanged,
        # so a sound judge should see a rationale that argues the opposite of its conclusion.
        return re.sub(r"[A-Za-z]+",
                      lambda m: (_flip[m.group(0).lower()].capitalize()
                                 if m.group(0)[:1].isupper() and m.group(0).lower() in _flip
                                 else _flip.get(m.group(0).lower(), m.group(0))),
                      txt)

    variants = []
    for _, r in gen.iterrows():
        base = dict(case_id=r["case_id"], model_tag=r["model_tag"],
                    fin_profile=r.get("fin_profile", ""), context=r.get("context", ""),
                    delta=r.get("delta", 0), gen_condition=r.get("gen_condition", "gap_given"))
        rat, txt = r["gen_rating"], str(r["gen_rationale"])
        variants.append({**base, "ptype": "P0_original", "level": 0,
                         "rating": rat, "rationale": txt})
        # P1: rating-rationale inconsistency (shift the grade, keep the text).
        for lv in (1, 2, 3):
            variants.append({**base, "ptype": "P1_inconsistency", "level": lv,
                             "rating": shift_rating(rat, lv), "rationale": txt})
        # P2: delete the qualitative-support clause (grade kept). Heuristic sentence drop.
        sents = [s for s in txt.replace("\n", " ").split(". ") if s]
        kept = [s for s in sents if not any(k in s.lower() for k in SUPPORT_KW)]
        variants.append({**base, "ptype": "P2_remove_qualitative", "level": 1,
                         "rating": rat, "rationale": ". ".join(kept)})
        # P3: inject a factual (quantitative) error that contradicts the financial input.
        p3 = (txt + " The firm reports negative shareholders' equity and a current ratio far "
              "below 1.0, indicating acute liquidity stress.")
        variants.append({**base, "ptype": "P3_factual_error", "level": 1,
                         "rating": rat, "rationale": p3})
        # P4: direction flip (reverse support polarity words; grade unchanged).
        variants.append({**base, "ptype": "P4_direction_flip", "level": 1,
                         "rating": rat, "rationale": flip_direction(txt)})
        # P5: verbosity control (pad with repetition; content unchanged).
        variants.append({**base, "ptype": "P5_verbosity", "level": 1,
                         "rating": rat, "rationale": (txt + " ") * 3})
    pert = pd.DataFrame(variants)
    write_df(pert, PROC / "perturbations")
    save_table(pert["ptype"].value_counts().rename_axis("ptype").reset_index(name="n"),
               "04_perturbation_counts")
    print("wrote perturbations:", len(pert), "rows")


wrote perturbations: 2400 rows
